# ADE Classifier — GPU training runs

All non-GPU code lives in `src/pv_ade/`. This notebook only calls into that package and writes per-run metrics + raw predictions to `results/`. See `PLAN.md` for the ablation design.

The cells are designed so "Run All" is safe regardless of project state — the sweep and ablation cells both consult `configs/ablation.yaml` and skip themselves when not applicable.

Workflow:
1. Make the repo available (GitHub clone or Drive mount — toggle with `SETUP_MODE`).
2. Install dependencies.
3. Generate the split if it doesn't exist yet (committed after the first time).
4. Run the per-model LR sweep on val if any entry in `training.learning_rate_per_model` is null, then auto-pick and persist a winner per model.
5. Run the final ablation (3 models × N seeds) once every model has a locked LR.
6. Download `results/` and commit.

## 1. Repo access

Set `SETUP_MODE = "github"` (default) to clone the repo fresh from GitHub, or `"drive"` to mount Drive and use a folder you've uploaded there at `MyDrive/projects/pharmacovigilance-ade-classifier/`.

In [ ]:
SETUP_MODE = "github"  # "github" | "drive"

REPO_URL = "https://github.com/tjohns94/pharmacovigilance-ade-classifier.git"
DRIVE_PATH = "/content/drive/MyDrive/projects/pharmacovigilance-ade-classifier"

if SETUP_MODE == "github":
    !git clone $REPO_URL /content/portfolio
    %cd /content/portfolio
elif SETUP_MODE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    %cd $DRIVE_PATH
else:
    raise ValueError(f"Unknown SETUP_MODE: {SETUP_MODE!r}")

## 2. Install dependencies + wire up the package

Uses `sys.path` rather than `pip install -e .` because editable installs in a src-layout project are flaky in Colab without a kernel restart.

In [ ]:
!pip install -q -r requirements-colab.txt

import os, sys
sys.path.insert(0, os.path.abspath('src'))

## 3. Load config + ensure split exists

Splits are committed to the repo. This cell regenerates them only if missing.

In [ ]:
from pathlib import Path
import yaml

from pv_ade.data import generate_splits, load_ade_corpus, load_splits

with open('configs/ablation.yaml') as f:
    cfg = yaml.safe_load(f)
splits_path = Path(cfg['dataset']['splits_path'])

if not splits_path.exists():
    raw = load_ade_corpus()
    generate_splits(
        raw,
        seed=cfg['dataset']['split_seed'],
        ratios=tuple(cfg['dataset']['split_ratios']),
        out_path=splits_path,
    )
splits = load_splits(splits_path)
{k: len(v) for k, v in splits.items()}

## 4. Per-model LR sweep (val set only)

Runs when any entry in `training.learning_rate_per_model` is null in `configs/ablation.yaml`. For each model with a missing winner, trains one run per candidate LR at `seeds[0]`, evaluates on val, and picks the winner on `evaluation.primary_metric` (default `macro_f1`). Winners are written back to `configs/ablation.yaml` so future sessions skip straight to the ablation.

A per-model sweep is used (rather than sweeping one model and reusing its LR) to verify that hyperparameters transfer across pretraining backbones. Writing a winner per model lets the final ablation use a different LR for each backbone, so a weaker model isn't penalized by another model's hyperparameter choice — which would otherwise bias the between-model comparison.

In [ ]:
import json
import math
import shutil

from pv_ade.train import train_one_run

lrpm = cfg['training']['learning_rate_per_model']
if all(v is not None for v in lrpm.values()):
    print(
        f"Sweep already locked — skipping. learning_rate_per_model={lrpm}. "
        "Null any entry in configs/ablation.yaml to re-run the sweep."
    )
else:
    sweep_seed = cfg['training']['seeds'][0]
    sweep_epochs = int(cfg['sweep']['num_epochs'])
    sweep_lrs = [float(lr) for lr in cfg['sweep']['learning_rates']]
    sweep_models = cfg['sweep']['models']
    sweep_metrics_dir = Path('results/sweep/metrics')
    sweep_predictions_dir = Path('results/sweep/predictions')
    primary = cfg['evaluation']['primary_metric']

    # Fresh sweep — clear stale results so the winner-picker can't pick up runs
    # from a previous sweep topology (different LR grid, different models, ...).
    for d in (sweep_metrics_dir, sweep_predictions_dir):
        if d.exists():
            shutil.rmtree(d)

    for model_name in sweep_models:
        model_spec = next(m for m in cfg['models'] if m['name'] == model_name)
        for lr in sweep_lrs:
            run_cfg = {
                **{k: v for k, v in cfg['training'].items() if k != 'learning_rate_per_model'},
                'learning_rate': lr,
                'num_epochs': sweep_epochs,
            }
            print(f'sweep: model={model_name}, lr={lr}, epochs={sweep_epochs}')
            train_one_run(
                model_name=f"{model_name}_lr{lr}",
                checkpoint=model_spec['checkpoint'],
                seed=sweep_seed,
                config=run_cfg,
                splits_path=splits_path,
                metrics_dir=sweep_metrics_dir,
                predictions_dir=sweep_predictions_dir,
                eval_split='val',
            )

    # Pick per-model winners on primary_metric.
    per_model_best: dict[str, float] = {}
    for model_name in sweep_models:
        candidates = []
        for lr in sweep_lrs:
            mf = sweep_metrics_dir / f"{model_name}_lr{lr}_seed{sweep_seed}.json"
            rec = json.loads(mf.read_text())
            candidates.append({'lr': rec['config']['learning_rate'], 'score': rec['metrics'][primary]})
        best = max(candidates, key=lambda r: r['score'])
        per_model_best[model_name] = best['lr']
        print(f"winner {model_name}: lr={best['lr']} (val {primary}={best['score']:.4f})")

    # Update in-memory config so the ablation cell can proceed in this runtime.
    cfg['training']['learning_rate_per_model'] = dict(per_model_best)

    # Persist winners to configs/ablation.yaml. Line-based edit scoped to the
    # learning_rate_per_model block to preserve surrounding comments and
    # formatting.
    cfg_path = Path('configs/ablation.yaml')
    lines = cfg_path.read_text().splitlines(keepends=True)
    in_lrpm = False
    for i, line in enumerate(lines):
        stripped = line.lstrip()
        if not stripped or stripped.startswith('#'):
            continue
        indent = len(line) - len(stripped)
        if indent == 2 and stripped.startswith('learning_rate_per_model:'):
            in_lrpm = True
            continue
        if in_lrpm and indent <= 2:
            in_lrpm = False
        if in_lrpm and indent == 4:
            name = stripped.split(':', 1)[0].strip()
            if name in per_model_best:
                lines[i] = f"    {name}: {per_model_best[name]:.1e}\n"
    cfg_path.write_text(''.join(lines))

    # Reload-and-assert: line-based YAML patching is fragile, so verify both
    # that every model got written and that the values round-trip within the
    # precision of `.1e` formatting.
    reloaded = yaml.safe_load(cfg_path.read_text())['training']['learning_rate_per_model']
    for name, expected in per_model_best.items():
        got = reloaded.get(name)
        assert got is not None and math.isclose(got, expected, rel_tol=0.05), (
            f'learning_rate_per_model[{name}] reload mismatch: wrote {expected}, read {got}'
        )
    print('configs/ablation.yaml updated on disk and reload check passed.')

## 5. Final ablation — 3 models × N seeds

Runs only if every entry in `training.learning_rate_per_model` is set. With those locked, you can "Run All" from a fresh runtime and this cell will execute after the sweep is correctly skipped.

In [ ]:
per_model_lr = cfg['training']['learning_rate_per_model']
missing = [name for name, v in per_model_lr.items() if v is None]
if missing or cfg['training']['num_epochs'] is None:
    print(
        'Ablation skipped — missing per-model learning rates ('
        + ', '.join(missing) + ') or num_epochs in configs/ablation.yaml. '
        'Run the sweep cell first.'
    )
else:
    metrics_dir = Path(cfg['output']['metrics_dir'])
    predictions_dir = Path(cfg['output']['predictions_dir'])

    for model_spec in cfg['models']:
        model_lr = float(per_model_lr[model_spec['name']])
        run_cfg = {
            **{k: v for k, v in cfg['training'].items() if k != 'learning_rate_per_model'},
            'learning_rate': model_lr,
        }
        for seed in cfg['training']['seeds']:
            print(f"=== {model_spec['name']} seed={seed} lr={model_lr} ===")
            train_one_run(
                model_name=model_spec['name'],
                checkpoint=model_spec['checkpoint'],
                seed=seed,
                config=run_cfg,
                splits_path=splits_path,
                metrics_dir=metrics_dir,
                predictions_dir=predictions_dir,
                eval_split='test',
            )

## 6. Zip + download results

Then commit `results/` from the local repo.

In [ ]:
import shutil
shutil.make_archive('/content/results', 'zip', 'results')
from google.colab import files
files.download('/content/results.zip')